In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=8

In [3]:
# 1. Load
grav = pd.read_csv(DATA / 'Gravity_csv_V202211' / 'Gravity_V202211.csv', usecols=["iso3_o", "iso3_d", "year", "distw_harmonic", "dist", "contig"], low_memory=False)

In [4]:
# 2. Pick the most recent year (or any reference year)
g = grav[grav.year == 2021].copy()

In [5]:
# 3. Diagnose duplicates (CEPII 有些 ISO3 对应多个内部 country_id)
dups = g.groupby(["iso3_o", "iso3_d"]).size()
print(f"重复的 (origin, destination) 对: {(dups > 1).sum()}")

重复的 (origin, destination) 对: 4293


In [6]:
# 4. Pivot — 用 pivot_table + mean 处理重复
dist_matrix = g.pivot_table(
    index="iso3_o",
    columns="iso3_d",
    values="distw_harmonic",
    aggfunc="mean"
)

In [7]:
# 5. 限制到你的国家列表 + 对齐行列顺序
country_list = sorted(set(g.iso3_o) & set(g.iso3_d))  # 或换成你的 ~190 国家列表
dist_matrix = dist_matrix.reindex(index=country_list, columns=country_list)

In [8]:
# 6. 对角线设为 NaN(避免自我贸易)
np.fill_diagonal(dist_matrix.values, np.nan)

In [9]:
dist_matrix

iso3_d,ABW,AFG,AGO,AIA,ALB,AND,ANT,ARE,ARG,ARM,...,VNM,VUT,WLF,WSM,YEM,YMD,YUG,ZAF,ZMB,ZWE
iso3_o,,,,,,,,,,,,,,,,,,,,,
ABW,NaN,13175.0,9575.0,978.0,9090.0,7570.0,NaN,12772.0,5145.0,11109.0,...,17013.0,13759.0,12052.0,11591.0,12222.0,NaN,NaN,11317.0,11245.0,11538.0
AFG,13175.0,NaN,7509.0,12212.0,4240.0,5712.0,NaN,1598.0,15278.0,2152.0,...,4463.0,12036.0,13198.0,13634.0,3219.0,NaN,NaN,8185.0,6861.0,7070.0
AGO,9575.0,7509.0,NaN,8973.0,5669.0,5897.0,NaN,5910.0,7947.0,6358.0,...,10574.0,15895.0,17243.0,17334.0,4324.0,NaN,NaN,2507.0,1682.0,2046.0
AIA,978.0,12212.0,8973.0,NaN,8112.0,6592.0,NaN,11798.0,5686.0,10134.0,...,16231.0,14610.0,12882.0,12435.0,11302.0,NaN,NaN,10939.0,10656.0,10995.0
ALB,9090.0,4240.0,5669.0,8112.0,NaN,1521.0,NaN,3724.0,11628.0,2082.0,...,8704.0,16037.0,16535.0,16759.0,3811.0,NaN,NaN,7761.0,6255.0,6707.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YMD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
YUG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ZAF,11317.0,8185.0,2507.0,10939.0,7761.0,8281.0,NaN,6635.0,8051.0,7809.0,...,9762.0,13414.0,14741.0,14888.0,5100.0,NaN,NaN,NaN,1469.0,1049.0


In [10]:
# 8. 检查
print(f"矩阵形状: {dist_matrix.shape}")
print(f"缺失值数量: {dist_matrix.isna().sum().sum()} (含对角线 {len(country_list)})")
print(f"对称性误差 (应为 0): {(dist_matrix - dist_matrix.T).abs().max().max():.6f}")

矩阵形状: (243, 243)
缺失值数量: 4059 (含对角线 243)
对称性误差 (应为 0): 0.000000


In [11]:
dist_matrix.to_csv(RESULTS / 'bilateral-distance-matrix.csv')

In [12]:
country_pseudo_values = pd.read_csv(DATA / 'country-pseudo-values.csv')

In [13]:
country_pseudo_values

,value,country,full name,region,GRIP_region,notes
0,1,IRN,Iran,Middle East,5.0,NaN
1,2,BLM,Saint-Barth??lemy,Latin America and the Caribbean,2.0,NaN
2,3,UGA,Uganda,East Africa,3.0,NaN
3,4,SLE,Sierra Leone,West Africa,3.0,NaN
4,5,VCT,Saint Vincent and the Grenadines,Latin America and the Caribbean,2.0,NaN
...,...,...,...,...,...,...
250,251,MNP,Northern Mariana Islands,Oceania,7.0,NaN
251,252,IOT,British Indian Ocean Territory,Southern Africa,3.0,NaN
252,253,ARM,Armenia,Central and Northern Asia,5.0,Western Asia
253,254,HKG,Hong Kong,Eastern Asia,6.0,NaN


In [14]:
dist_matrix.columns

Index(['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ANT', 'ARE', 'ARG', 'ARM',
       ...
       'VNM', 'VUT', 'WLF', 'WSM', 'YEM', 'YMD', 'YUG', 'ZAF', 'ZMB', 'ZWE'],
      dtype='object', name='iso3_d', length=243)

In [15]:
country_pseudo_values['country'].values

array(['IRN', 'BLM', 'UGA', 'SLE', 'VCT', 'BFA', 'SRB', 'TGO', 'BLZ',
       'MYS', 'SMR', 'ALA', 'VIR', 'BHS', 'SLB', 'MUS', 'ETH', 'POL',
       'MDV', 'AIA', 'TCD', 'HTI', 'SEN', 'NGA', 'TLS', 'CXR', 'EST',
       'PRY', 'AFG', 'LUX', 'WLF', 'GIN', 'ASM', 'ZWE', 'BGR', 'XCL',
       'ESH', 'AND', 'NIU', 'VNM', 'OMN', 'KNA', 'LVA', 'FRO', 'UMI',
       'AUT', 'ISL', 'DOM', 'DMA', 'LIE', 'VGB', 'LBR', 'EGY', 'ATF',
       'XKO', 'AGO', 'IRQ', 'MLI', 'BRA', 'SJM', 'HRV', 'MTQ', 'RWA',
       'LKA', 'ISR', 'BOL', 'MNG', 'MCO', 'LCA', 'WSM', 'NER', 'CMR',
       'TUR', 'QAT', 'TWN', 'BES', 'GNQ', 'CIV', 'TUV', 'NAM', 'KGZ',
       'BEL', 'DNK', 'PLW', 'MKD', 'NPL', 'TTO', 'MSR', 'SUR', 'CZE',
       'GUF', 'STP', 'MDA', 'USA', 'GEO', 'ALB', 'PCN', 'IND', 'MOZ',
       'GLP', 'DEU', 'BVT', 'BIH', 'TZA', 'TUN', 'PAK', 'JOR', 'CYP',
       'LSO', 'ZAF', 'SDN', 'MAF', 'MAR', 'FRA', 'IRL', 'CHN', 'FSM',
       'MEX', 'SXM', 'ARE', 'GBR', 'REU', 'MMR', 'IDN', 'MRT', 'BTN',
       'HUN', 'IMN',

In [16]:
cepii_iso3   = set(dist_matrix.columns)
my_iso3      = set(country_pseudo_values['country'].values)

# 三个集合
both         = cepii_iso3 & my_iso3
only_in_mine = my_iso3 - cepii_iso3   # ← 关键: 我有但 CEPII 没有
only_in_cepii = cepii_iso3 - my_iso3   # CEPII 有但我没用到 (无所谓)

print(f"CEPII: {len(cepii_iso3)}")
print(f"My country_raster: {len(my_iso3)}")
print(f"交集: {len(both)}")
print(f"我有 CEPII 没有的 (需处理): {len(only_in_mine)}")
print(f"   → {sorted(only_in_mine)}")
print(f"CEPII 有我没用到的: {len(only_in_cepii)}")

CEPII: 243
My country_raster: 255
交集: 235
我有 CEPII 没有的 (需处理): 20
   → ['ALA', 'ATA', 'ATF', 'BLM', 'BVT', 'GGY', 'HMD', 'IMN', 'JEY', 'MAF', 'SGS', 'SJM', 'UMI', 'VIR', 'XAD', 'XCL', 'XKO', 'XPI', 'XSP', 'ZNC']
CEPII 有我没用到的: 8


In [17]:
country_raster_1km = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif').read(1)

In [18]:
demand_raster = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\sand-demand-2023-2050-BAU-50km-M.tif").read(1)

In [19]:
country_pseudo_values

,value,country,full name,region,GRIP_region,notes
0,1,IRN,Iran,Middle East,5.0,NaN
1,2,BLM,Saint-Barth??lemy,Latin America and the Caribbean,2.0,NaN
2,3,UGA,Uganda,East Africa,3.0,NaN
3,4,SLE,Sierra Leone,West Africa,3.0,NaN
4,5,VCT,Saint Vincent and the Grenadines,Latin America and the Caribbean,2.0,NaN
...,...,...,...,...,...,...
250,251,MNP,Northern Mariana Islands,Oceania,7.0,NaN
251,252,IOT,British Indian Ocean Territory,Southern Africa,3.0,NaN
252,253,ARM,Armenia,Central and Northern Asia,5.0,Western Asia
253,254,HKG,Hong Kong,Eastern Asia,6.0,NaN


In [20]:
country_raster_1km

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(16922, 33844), dtype=uint8)

In [21]:
# 1. 加载 50km country raster (与 demand 分辨率对齐)
country_raster_50km = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif').read(1)

In [22]:
# 2. 加载三种材料的总需求 (用 BAU-M 作为代表性场景)
sand      = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\sand-demand-2023-2050-BAU-50km-M.tif").read(1)
gravel    = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\gravel-demand-2023-2050-BAU-50km-M.tif").read(1)
limestone = rio.open(r"F:\aggregates-mining-biodiversity\results\uncertainty-analysis\limestone-demand-2023-2050-BAU-50km-M.tif").read(1)

total_demand = sand + gravel + limestone  # 单位: t (per pixel)
global_total = total_demand.sum() / 1e9   # Gt

In [23]:
# 3. 检查 20 个缺失国家
only_in_mine = ['ALA', 'ATA', 'ATF', 'BLM', 'BVT', 'GGY', 'HMD', 'IMN', 
                'JEY', 'MAF', 'SGS', 'SJM', 'UMI', 'VIR', 'XAD', 'XCL', 
                'XKO', 'XPI', 'XSP', 'ZNC']

results = []
for iso in only_in_mine:
    row = country_pseudo_values[country_pseudo_values['country'] == iso]
    if row.empty:
        results.append((iso, None, 0, '⚠️ 未找到'))
        continue
    code = row['value'].values[0]
    full_name = row['full name'].values[0]
    mask = (country_raster_50km == code)
    pixel_count = mask.sum()
    d_gt = total_demand[mask].sum() / 1e9
    d_pct = 100 * d_gt / global_total
    results.append((iso, full_name, pixel_count, d_gt, d_pct))

In [24]:
# 4. 排序展示
df = pd.DataFrame(results, columns=['ISO3', 'name', 'pixels', 'demand_Gt', 'pct_of_global'])
df = df.sort_values('demand_Gt', ascending=False)
print(df.to_string(index=False))

print(f"\n全球总需求 (sand+gravel+limestone, BAU-M): {global_total:.2f} Gt")
print(f"20 国合计: {df['demand_Gt'].sum():.4f} Gt ({100*df['demand_Gt'].sum()/global_total:.3f}%)")

ISO3                             name  pixels  demand_Gt  pct_of_global
 ALA                           ??land       1        0.0            0.0
 ATA                       Antarctica    4860        0.0            0.0
 ATF      French Southern Territories       4        0.0            0.0
 BLM                Saint-Barth??lemy       0        0.0            0.0
 BVT                    Bouvet Island       0        0.0            0.0
 GGY                         Guernsey       0        0.0            0.0
 HMD Heard Island and McDonald Island       1        0.0            0.0
 IMN                      Isle of Man       0        0.0            0.0
 JEY                           Jersey       0        0.0            0.0
 MAF                     Saint-Martin       0        0.0            0.0
 SGS South Georgia and the South Sand       2        0.0            0.0
 SJM           Svalbard and Jan Mayen      24        0.0            0.0
 UMI United States Minor Outlying Isl       0        0.0        

## Calibrate trade propensities from historical bilateral trade

Pure gravity (`T_ij = S_i · D_j / d_ij^β`) cannot recover real-world aggregate trade patterns — it incorrectly elevates large-surplus economies (Russia, USA, Europe) to top exporters and overlooks the structural exporters (UAE, Malaysia, Norway, China) that dominate real trade.

We fix this with **country fixed effects** calibrated from historical UN Comtrade-style data:

$$T_{ij} = \alpha_i \cdot \beta_j \cdot S_i \cdot D_j \cdot d_{ij}^{-\beta_{\text{trade}}}$$

- **`α_i`** = country `i`'s historical *export* propensity (structural, time-invariant)
- **`β_j`** = country `j`'s historical *import* propensity (structural, time-invariant)
- **`S_i, D_j`** = future surplus / deficit (scenario-dependent — this carries the dynamics)

Calibrated as the **share of global trade**, normalized so `mean(α) = mean(β) = 1.0`, floored at 0.1.

Inputs: `results/trade-data/{other-sand, gravel-crushed-stone, limestone}-{export, import}-142.csv` (unit: kt, 2018-2022 averaged).

Output: `results/trade-propensities.csv` — consumed by `5-2-map-extraction-BAU.ipynb`.

In [25]:
# Combine sand (post-1988 = 'other-sand'), gravel-crushed-stone, and limestone.
# Note: 'sand-export-142.csv' has data only up to 1988 (HS code reclassification).
TRADE_DATA  = RESULTS / 'trade-data'
YEAR_WINDOW = list(range(2018, 2023))
CATS        = ['other-sand', 'gravel-crushed-stone', 'limestone']

exp_total = sum(
    pd.read_csv(TRADE_DATA / f'{cat}-export-142.csv', index_col=0).loc[YEAR_WINDOW].mean()
    for cat in CATS
)
imp_total = sum(
    pd.read_csv(TRADE_DATA / f'{cat}-import-142.csv', index_col=0).loc[YEAR_WINDOW].mean()
    for cat in CATS
)
# Values in kt; convert to Mt for readability of totals.
print(f'Historical aggregate trade (2018-2022 avg):')
print(f'  Total exports: {exp_total.sum() / 1e3:.1f} Mt/yr')
print(f'  Total imports: {imp_total.sum() / 1e3:.1f} Mt/yr  (should match exports)')
print(f'  Countries with > 0 exports: {(exp_total > 0).sum()}')
print(f'  Countries with > 0 imports: {(imp_total > 0).sum()}')

Historical aggregate trade (2018-2022 avg):
  Total exports: 318.7 Mt/yr
  Total imports: 318.7 Mt/yr  (should match exports)
  Countries with > 0 exports: 137
  Countries with > 0 imports: 137


In [26]:
# Normalize so mean = 1.0; floor at 0.1 to avoid permanently excluding sparse-data countries.
alpha_export = (exp_total / exp_total.mean()).clip(lower=0.1)
beta_import  = (imp_total / imp_total.mean()).clip(lower=0.1)

propensities = pd.DataFrame({
    'alpha_export': alpha_export,
    'beta_import' : beta_import,
})
print(f'Calibrated propensities for {len(propensities)} countries')
print(propensities.describe().round(3))

Calibrated propensities for 142 countries
       alpha_export  beta_import
count       142.000      142.000
mean          1.061        1.057
std           3.056        2.701
min           0.100        0.100
25%           0.100        0.100
50%           0.100        0.100
75%           0.371        0.360
max          24.966       16.429


In [27]:
# Sanity check: top structural exporters and importers should match real-world intuition.
print('=== Top 15 by α (export propensity) — structural exporters ===')
print(propensities.nlargest(15, 'alpha_export').round(2))
print()
print('=== Top 15 by β (import propensity) — structural importers ===')
print(propensities.nlargest(15, 'beta_import').round(2))

=== Top 15 by α (export propensity) — structural exporters ===
     alpha_export  beta_import
ARE         24.97         2.44
CHN         15.41         1.26
MYS         12.26         0.10
OMN         10.02         0.21
NOR          9.96         0.10
DEU          8.24         7.06
SUN          6.05         7.78
BEL          5.84         7.78
CAN          4.92         0.75
NLD          4.28         8.93
FRA          4.27         4.73
IDN          3.87         0.57
MEX          3.65         0.12
GBR          2.99         1.39
YUG          2.05         1.60

=== Top 15 by β (import propensity) — structural importers ===
     alpha_export  beta_import
SGP          0.10        16.43
HKG          0.18        14.12
IND          1.89        11.55
USA          0.97         9.99
KWT          0.10         9.68
NLD          4.28         8.93
SUN          6.05         7.78
BEL          5.84         7.78
QAT          0.10         7.10
DEU          8.24         7.06
FRA          4.27         4.73
EWP  

In [28]:
# Align propensity set to the gravity-distance ISO set used in 5-2.
# Countries present in CEPII but missing from historical trade data → defaulted to 0.1 (the floor),
# i.e. assumed minimal trade activity. This is conservative: if there's no historical evidence of
# trade in/out of a country, the model treats it as a small player rather than imputing average behavior.
all_iso = sorted(set(propensities.index) | set(dist_matrix.columns))
propensities_aligned = propensities.reindex(all_iso).fillna(0.1)

out = RESULTS / 'trade-propensities.csv'
propensities_aligned.to_csv(out)
print(f'Saved: {out}')
print(f'  Shape: {propensities_aligned.shape}')
print(f'  Countries with historical data: {propensities.shape[0]}')
print(f'  Countries with default (0.1): {len(all_iso) - propensities.shape[0]}')

Saved: d:\wty\Documents\GitHub\aggregate-mining-biodiversity\results\trade-propensities.csv
  Shape: (245, 2)
  Countries with historical data: 142
  Countries with default (0.1): 103
